### Step 1: Clean and Prepare the Data

In [ ]:
import pandas as pd

# Use the full path to the file
file_path = '../data/groundwater_ml_dataset_final.csv'
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    # Fallback for different working directory
    df = pd.read_csv('data/groundwater_ml_dataset_final.csv')

# Ensure only relevant columns are kept if they exist
expected_columns = ['Lat', 'Lon', 'Data', 'Inundatie_Target', 'elevation', 'mean_precipitation_mm', 'soil_moisture']
df = df[[col for col in expected_columns if col in df.columns]]

# Now drop rows that have ANY missing values
df_cleaned = df.dropna()

print(f"Shape after cleaning: {df_cleaned.shape}")
df_cleaned.to_csv('../data/better_cleaned_groundwater_dataset.csv', index=False)

### Step 2: Train the Random Forest Model

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================
# 1. LOAD DATA 
# ==========================================
try:
    df = pd.read_csv('../data/finalDataset.csv')
except FileNotFoundError:
    df = pd.read_csv('data/finalDataset.csv')

# ==========================================
# 2. DEFINE FEATURES AND TARGET
# ==========================================
# Features based on the provided CSV structure
features = ['elevation', 'mean_precipitation_mm', 'soil_moisture']
target = 'Inundatie_Target'

X = df[features]
y = df[target]

# ==========================================
# 3. SPLIT DATA (TRAINING & VALIDATION)
# ==========================================
X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y 
)

print(f"Total samples: {len(df)}")
print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}\n")

# ==========================================
# 4. INITIALIZE AND TRAIN THE MODEL
# ==========================================
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

print("Training the Random Forest model...")
rf_model.fit(X_train, y_train)

# ==========================================
# 5. PREDICT AND EVALUATE
# ==========================================
print("Evaluating on the validation set...\n")

y_pred = rf_model.predict(X_val)

y_probs = rf_model.predict_proba(X_val)
confidences = np.max(y_probs, axis=1)

accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_val, y_pred))

print("Feature Importances:")
importances = rf_model.feature_importances_
for feature, importance in zip(features, importances):
    print(f" - {feature}: {importance:.4f}")

print("\nSample Validation Results with Confidence:")
val_results = X_val.copy()
val_results['actual'] = y_val.values
val_results['predicted'] = y_pred
val_results['confidence'] = confidences
print(val_results.head())

Total samples: 287
Training samples: 229
Validation samples: 58

Training the Random Forest model...
Evaluating on the validation set...

Validation Accuracy: 79.31%

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.81      0.79        27
           1       0.83      0.77      0.80        31

    accuracy                           0.79        58
   macro avg       0.79      0.79      0.79        58
weighted avg       0.80      0.79      0.79        58

Feature Importances:
 - elevation: 0.3417
 - mean_precipitation_mm: 0.3833
 - soil_moisture: 0.2751

Sample Validation Results with Confidence:
       elevation  mean_precipitation_mm  soil_moisture  actual  predicted  \
235   415.273438               0.024586      91.007507       1          0   
217    98.611443               1.695015      61.891251       1          1   
183   114.927750               7.354743      87.912422       1          1   
173   772.098083               